# PipeLens Tutorial: Explaining and Repairing a Failing Pipeline

This notebook is a step-by-step tutorial for the code repository.  
It demonstrates how to:

1. How to log historical executions on the passing/training data.
2. Select the optimized pipeline from the historical executions.
3. Run the same pipeline on failing/test data and observe the malfunction.
4. Construct profiles for the failing data.
5. Rank candidate interventions in the glass-box setting.
6. Evaluate ranked interventions until the utility goal is reached.

The example uses the regression setting by default:
`dataset = hmda`, `metric = accuracy_score`, and `model = lr`.

Feel free to explore other datasets.

## Imports

This cell imports the algorithm modules.

In [25]:
import os
import json
import random
import logging
import numpy as np
import pandas as pd

from pipeline_execution import PipelineExecutor
from glassbox_optimizer import GlassBoxOptimizer

np.random.seed(42)
random.seed(42)

## Cell 2 — Load configuration

In [26]:
CONFIG_PATH = "config_example.json"

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r") as f:
        config = json.load(f)
    print("Loaded config_example.json")
else:
    config = {
        "method": "PipeLens",
        "dataset_name": "hmda",
        "model_type": "lr",
        "metric_type": "accuracy_score",
        "pipeline_type": "ml",
        "pipeline_order": ["sampling","invalid_value","missing_value","floating_point","normalization","model"],
        "new_components": ["outlier", "deduplication"],
        "f_goals": {"hmda": [0.1]},
        "paths": {
            "train_data": "historical_data/tutorial/train_profile_{model_type}_{metric_type}_{dataset_name}.csv",
            "test_data": "historical_data/tutorial/test_profile_{model_type}_{metric_type}_{dataset_name}.csv",
            "metric_output": "historical_data/tutorial/{method}_{model_type}_{metric_type}_{dataset_name}_result.csv"
        }
    }

method = config.get("method", "PipeLens")
dataset_name = config.get("dataset_name", "hmda")
model_type = config.get("model_type", "nn")
metric_type = config.get("metric_type", "accuracy_score")
pipeline_type = config.get("pipeline_type", "ml")
pipeline_order = config.get("pipeline_order", ["missing_value", "normalization", "outlier", "model"])
new_components = config.get("new_components", ["fselection"])

utility_goals = config.get("f_goals", {}).get(dataset_name, [0.1])
f_goal = utility_goals[0]

print("Dataset:", dataset_name)
print("Model:", model_type)
print("Metric:", metric_type)
print("Pipeline order:", pipeline_order)
print("Candidate new components:", new_components)
print("Tutorial utility goal:", f_goal)

Dataset: hmda
Model: lr
Metric: accuracy_score
Pipeline order: ['sampling', 'invalid_value', 'missing_value', 'floating_point', 'normalization', 'model']
Candidate new components: ['outlier', 'deduplication']
Tutorial utility goal: 0.1


## Tutorial paths

The tutorial writes its own historical profile files under `historical_data/tutorial/`.

In [27]:
os.makedirs("historical_data/tutorial", exist_ok=True)
os.makedirs("logs/tutorial", exist_ok=True)

filename_train = f"historical_data/tutorial/train_profile_{model_type}_{metric_type}_{dataset_name}.csv"
filename_test_current = f"historical_data/tutorial/test_current_profile_{model_type}_{metric_type}_{dataset_name}.csv"
filename_rank = "fused_ranking.csv"

logging.basicConfig(
    filename=f"logs/tutorial/pipelens_tutorial_{dataset_name}_{model_type}_{metric_type}.log",
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

print("Historical train profile path:", filename_train)
print("Current failing profile path:", filename_test_current)

Historical train profile path: historical_data/tutorial/train_profile_lr_accuracy_score_hmda.csv
Current failing profile path: historical_data/tutorial/test_current_profile_lr_accuracy_score_hmda.csv


## Helper functions

These helpers make the notebook easier to read.

Important convention:
- Pipeline strategy vectors are stored as Encoded(**1-based indices**) in CSV files.
- All utility function is written in such a way that the lower is better.

In [28]:
LOWER_IS_BETTER = metric_type.lower() in {"rmse", "mse", "mae", "loss", "error", "accuracy_score", "sp"}

def passes_goal(value, goal):
    if LOWER_IS_BETTER:
        return value <= goal
    return value >= goal

def better_than(a, b):
    if LOWER_IS_BETTER:
        return a < b
    return a > b

def utility_sort_ascending():
    return LOWER_IS_BETTER

def get_utility_col(metric_type):
    return f"utility_{metric_type}"

def select_best_pipeline(df, pipeline_order, utility_col, goal):
    """Select a high-quality pipeline from historical passing executions."""
    df_valid = df.dropna(subset=[utility_col]).copy()

    if LOWER_IS_BETTER:
        candidates = df_valid[df_valid[utility_col] <= goal]
        if len(candidates) == 0:
            print("No historical row reaches the goal; selecting the minimum-utility row.")
            best_row = df_valid.loc[df_valid[utility_col].idxmin()]
        else:
            best_row = candidates.loc[candidates[utility_col].idxmin()]
    else:
        candidates = df_valid[df_valid[utility_col] >= goal]
        if len(candidates) == 0:
            print("No historical row reaches the goal; selecting the maximum-utility row.")
            best_row = df_valid.loc[df_valid[utility_col].idxmax()]
        else:
            best_row = candidates.loc[candidates[utility_col].idxmax()]

    best_param = [int(best_row[c]) for c in pipeline_order]
    return best_param, best_row

def show_pipeline_vector(order, vector):
    return " → ".join([f"{comp}:{strategy}" for comp, strategy in zip(order, vector)])

print("Lower metric is better:", LOWER_IS_BETTER)

Lower metric is better: True


## Log the historical execution from the passing/training dataset and find the optimize configuration.

We execute many randomly sampled pipeline configurations on the passing/training split and store:
- the pipeline strategy vector,
- intermediate data profiles,
- final utility.

For a quick tutorial, `n=100` is usually enough.

In [29]:
N_HISTORICAL = 100   

pass_executor = PipelineExecutor(
    pipeline_type=pipeline_type,
    dataset_name=dataset_name,
    metric_type=metric_type,
    pipeline_ord=pipeline_order,
    execution_type="pass"
)

df_train_profile = pass_executor.run_pipeline_glass_sample(
    filename_train,
    n=N_HISTORICAL
)

print("Historical profile data shape:", df_train_profile.shape)
display(df_train_profile.head())

[GlassRandom] File already exists -> skipping generation: historical_data/tutorial/train_profile_lr_accuracy_score_hmda.csv
Historical profile data shape: (100, 25)


,sampling,invalid_value,missing_value,floating_point,normalization,model,outlier_bef_normalization_strat,class_imbalance_ratio,corr_race,ot_race,...,ot_applicant_age,corr_lien_status,ot_lien_status,corr_LV,ot_LV,corr_DI,ot_DI,corr_income_brackets,ot_income_brackets,utility_accuracy_score
0,1,4,6,2,3,1,0.411284,8.846743,12472.04,0.34,...,0.34,18304.32,0.02,7663.49,0.37,3831.50,0.60,1089.89,0.27,0.098054
1,4,1,7,1,1,1,0.459195,7.365385,708.57,1.00,...,0.32,264.20,0.02,9056.22,0.51,6041.57,0.59,2611.16,0.31,0.115517
2,3,4,7,4,4,1,0.379377,7.426230,76.67,0.34,...,0.36,641.60,0.02,148.76,0.41,432.77,0.58,331.31,0.30,0.120623
3,4,3,2,4,3,1,0.459195,7.365385,3794.76,1.00,...,0.32,10730.31,0.02,4534.10,0.51,4488.55,0.59,3505.82,0.31,0.116092
4,3,2,2,4,4,1,0.379377,7.426230,76.67,0.34,...,0.36,641.60,0.02,148.76,0.41,432.77,0.58,331.31,0.30,0.120623


## Inspect utility distribution in historical executions

This cell shows the range of utilities observed in the historical passing executions.
All utility function is re-written so that lower values are better.

In [30]:
utility_col = get_utility_col(metric_type)

df_train_profile = pd.read_csv(filename_train)
print("Utility column:", utility_col)

summary = df_train_profile[utility_col].describe()
display(summary.to_frame(name="historical utility summary"))

if LOWER_IS_BETTER:
    print("Best historical utility:", df_train_profile[utility_col].min())
else:
    print("Best historical utility:", df_train_profile[utility_col].max())

Utility column: utility_accuracy_score


,historical utility summary
count,100.000000
mean,0.109095
std,0.009524
min,0.097276
25%,0.099870
50%,0.115517
75%,0.118677
max,0.120623


Best historical utility: 0.0972762645914396


## Select the best pipeline over the passing/training data

In [31]:
best_param, best_row = select_best_pipeline(
    df_train_profile,
    pipeline_order,
    utility_col,
    f_goal
)

print("Selected passing pipeline vector:")
print(show_pipeline_vector(pipeline_order, best_param))
print()
print("Passing-data utility:", best_row[utility_col])
print("Passes goal on passing data?", passes_goal(best_row[utility_col], f_goal))

display(best_row[pipeline_order + [utility_col]].to_frame(name="selected value"))

Selected passing pipeline vector:
sampling:2 → invalid_value:1 → missing_value:4 → floating_point:1 → normalization:2 → model:1

Passing-data utility: 0.0972762645914396
Passes goal on passing data? True


,selected value
sampling,2.000000
invalid_value,1.000000
missing_value,4.000000
floating_point,1.000000
normalization,2.000000
model,1.000000
utility_accuracy_score,0.097276


## Run the selected pipeline on failing/test data

Now we run the same optimized pipeline on the failing data.  
This demonstrates the core malfunction: the pipeline was good on passing data but fails on the new dataset.

In [32]:
fail_executor = PipelineExecutor(
    pipeline_type=pipeline_type,
    dataset_name=dataset_name,
    metric_type=metric_type,
    pipeline_ord=pipeline_order,
    execution_type="fail"
)

fixed_failing_data = fail_executor.get_injected_data()

initial_fail_utility = fail_executor.current_par_lookup(
    pipeline_order,
    best_param,
    fixed_data=fixed_failing_data
)

print("Selected pipeline:")
print(show_pipeline_vector(pipeline_order, best_param))
print()
print("Utility on failing data:", initial_fail_utility)
print("Utility goal:", f_goal)
print("Fails goal on failing data?", not passes_goal(initial_fail_utility, f_goal))

Selected pipeline:
sampling:2 → invalid_value:1 → missing_value:4 → floating_point:1 → normalization:2 → model:1

Utility on failing data: 0.13471502590673579
Utility goal: 0.1
Fails goal on failing data? True


## Save the pipeline

In [33]:
initial_order_ = list(pipeline_order)
initial_params_ = list(best_param)
initial_fail_utility_saved = initial_fail_utility

current_order = list(initial_order_)
current_params = list(initial_params_)
current_utility = initial_fail_utility_saved

print("initial pipeline:")
print(show_pipeline_vector(initial_order_, initial_params_))
print("initial failing utility:", initial_fail_utility_saved)

initial pipeline:
sampling:2 → invalid_value:1 → missing_value:4 → floating_point:1 → normalization:2 → model:1
initial failing utility: 0.13471502590673579


## Build a profile row for the current failing pipeline

The glass-box setting can inspect intermediate outputs.  
This helper executes the current pipeline on the fixed failing data and records profile features.

In [34]:
from modules.profiling.profile import Profile

def build_profile_row(executor, eval_order, eval_params, fixed_data, output_path=None):
    X, y, sens = fixed_data
    X, y, sens = X.copy(), y.copy(), sens.copy()

    profiler = Profile()
    numerical_columns = X.select_dtypes(include=["int", "float"]).columns

    param_record = []
    frac_data = []
    frac_headers = []
    utility = None
    fraction_outlier = None
    last_handler = None

    for i, step in enumerate(eval_order):
        param_index = executor._safe_param_index(step, int(eval_params[i]))
        handler = executor._load_handler(step, param_index)
        last_handler = handler

        X, y, sens, util_tmp, fraction_outlier, frac_header, frac_value = executor._apply_step(
            handler, X, y, sens
        )

        if frac_header is not None:
            frac_headers.append(frac_header)
            frac_data.append(frac_value)

        if util_tmp is not None:
            utility = util_tmp

        param_record.append(param_index + 1)

    headers, sens_data = last_handler.get_profile_metric(y, sens)
    profile_gen, key_profile = profiler.populate_profiles(
        pd.concat([X, y], axis=1),
        numerical_columns,
        executor.target_variable_name,
        fraction_outlier,
        executor.metric_type
    )

    cols = list(eval_order) + frac_headers + headers + key_profile + [get_utility_col(executor.metric_type)]
    row = param_record + frac_data + sens_data + profile_gen + [utility]
    row_df = pd.DataFrame([row], columns=cols)

    if output_path is not None:
        row_df.to_csv(output_path, index=False)

    return row_df

df_fail_current_profile = build_profile_row(
    fail_executor,
    pipeline_order,
    best_param,
    fixed_failing_data,
    output_path=filename_test_current
)

print("Current failing profile row shape:", df_fail_current_profile.shape)
print("Profile:")
print(df_fail_current_profile.head())

Current failing profile row shape: (1, 25)
Profile:
   sampling  invalid_value  missing_value  floating_point  normalization  \
0         2              1              4               1              2   

   model  outlier_bef_normalization_strat  class_imbalance_ratio  corr_race  \
0      1                         0.409326               5.892857     124.85   

   ot_race  ...  ot_applicant_age  corr_lien_status  ot_lien_status  corr_LV  \
0      0.3  ...              0.34            124.85            0.02   124.85   

   ot_LV  corr_DI  ot_DI  corr_income_brackets  ot_income_brackets  \
0   0.37   124.85   0.22                124.85                0.23   

   utility_accuracy_score  
0                0.134715  

[1 rows x 25 columns]


## Compare passing profiles and failing profiles

This cell provides a human-readable explanation of why the selected pipeline may fail. 

In [35]:
df_train_profile = pd.read_csv(filename_train)
df_fail_current_profile = pd.read_csv(filename_test_current)

strategy_cols = set(pipeline_order)
non_profile_cols = strategy_cols | {utility_col}
profile_cols = [
    c for c in df_train_profile.columns
    if c not in non_profile_cols and c in df_fail_current_profile.columns
]

if LOWER_IS_BETTER:
    successful_train = df_train_profile[df_train_profile[utility_col] <= f_goal]
else:
    successful_train = df_train_profile[df_train_profile[utility_col] >= f_goal]

if len(successful_train) == 0:
    successful_train = df_train_profile.copy()

train_profile_mean = successful_train[profile_cols].apply(pd.to_numeric, errors="coerce").mean()
fail_profile = df_fail_current_profile.iloc[0][profile_cols].apply(pd.to_numeric, errors="coerce")

profile_diff = pd.DataFrame({
    "profile": profile_cols,
    "passing_mean": train_profile_mean.values,
    "failing_current": fail_profile.values,
})
profile_diff["abs_difference"] = (profile_diff["passing_mean"] - profile_diff["failing_current"]).abs()
profile_diff = profile_diff.sort_values("abs_difference", ascending=False)

display(profile_diff.head(15))

,profile,passing_mean,failing_current,abs_difference
10,corr_lien_status,4959.840426,124.850000,4834.990426
12,corr_LV,3231.631702,124.850000,3106.781702
14,corr_DI,2654.913404,124.850000,2530.063404
4,corr_gender,2431.414681,124.850000,2306.564681
2,corr_race,2207.485319,124.850000,2082.635319
6,corr_loan_type,1907.744681,124.850000,1782.894681
16,corr_income_brackets,1549.957660,124.850000,1425.107660
8,corr_applicant_age,1546.264043,124.850000,1421.414043
1,class_imbalance_ratio,8.935171,5.892857,3.042314
5,ot_gender,1.000000,0.360000,0.640000


## Rank glass-box interventions

PipeLens now ranks candidate interventions using historical executions and the current failing pipeline.

The ranking considers:
- parameter changes to existing module,
- structural insertions from the global module catalog,
- profile similarity to successful passing executions,
- predicted utility from the historical proxy model.

In [40]:
glass = GlassBoxOptimizer(
    dataset_name,
    model_type,
    metric_type,
    pipeline_type,
    pipeline_order,
    filename_train,
    filename_test_current,
    new_components
)


cur_par = best_param.copy()

if hasattr(glass, "evaluate_interventions_pred_and_similarity"):
    fused_ranking = glass.evaluate_interventions_pred_and_similarity(
        cur_par,
        filename_train,
        new_components
    )
elif hasattr(fail_executor, "evaluate_interventions_pred_and_similarity"):
    fused_ranking = fail_executor.evaluate_interventions_pred_and_similarity(
        cur_par,
        filename_train,
        new_components
    )
else:
    raise AttributeError(
        "No evaluate_interventions_pred_and_similarity method found. "
        "Please expose the ranking method from GlassBoxOptimizer or PipelineExecutor."
    )

def ranking_to_dataframe(fused_ranking):
    rows = []
    for rank, item in enumerate(fused_ranking, start=1):
        component = item[0]
        strategy = item[1]
        similarity = item[2] if len(item) > 2 else np.nan
        utility_pred = item[3] if len(item) > 3 else np.nan
        position = item[4] if len(item) > 4 else None
        fused_score = item[5] if len(item) > 5 else np.nan

        intervention_type = "parameter change" if position is None else "structural insertion"
        rows.append({
            "rank": rank,
            "type": intervention_type,
            "component": component,
            "strategy": strategy,
            "position": position,
            "similarity": similarity,
            "predicted_utility": utility_pred,
            "fused_score": fused_score
        })
    return pd.DataFrame(rows)

df_rank_view = ranking_to_dataframe(fused_ranking)
display(df_rank_view.head(15))

[INFO] Profiles (features): ['class_imbalance_ratio', 'corr_race', 'ot_race', 'corr_gender', 'ot_gender', 'corr_loan_type', 'ot_loan_type', 'corr_applicant_age', 'ot_applicant_age', 'corr_lien_status', 'ot_lien_status', 'corr_LV', 'ot_LV', 'corr_DI', 'ot_DI', 'corr_income_brackets', 'ot_income_brackets']


,rank,type,component,strategy,position,similarity,predicted_utility,fused_score
0,1,structural insertion,outlier,2,3.0,0.550561,0.088404,0.500000
1,2,parameter change,missing_value,6,NaN,0.999836,0.089905,0.983509
2,3,parameter change,missing_value,8,NaN,0.999854,0.095023,0.927918
3,4,structural insertion,outlier,4,4.0,0.999816,0.102546,0.846132
4,5,parameter change,floating_point,4,NaN,0.999883,0.103649,0.834229
5,6,parameter change,normalization,5,NaN,0.606461,0.108458,0.344293
6,7,parameter change,missing_value,2,NaN,0.999904,0.110686,0.757790
7,8,parameter change,floating_point,2,NaN,0.999904,0.110767,0.756912
8,9,structural insertion,outlier,6,4.0,0.999864,0.113503,0.727140
9,10,parameter change,missing_value,3,NaN,0.999912,0.113848,0.723436


## Evaluate ranked interventions one by one

This cell follows the ranking and applies one intervention at a time.  
It keeps an intervention only if it improves the failing utility.

In [38]:
def apply_single_intervention(order, params, intervention):
    component, strategy, similarity, utility_pred, position, *rest = intervention

    new_order = list(order)
    new_params = list(params)

    if position is None:
        if component not in new_order:
            return None, None, "skip: component not in current pipeline"
        idx = new_order.index(component)
        new_params[idx] = int(strategy)
        action = f"change {component} to strategy {strategy}"
    else:
        pos = int(position)
        new_order = new_order[:pos] + [component] + new_order[pos:]
        new_params = new_params[:pos] + [int(strategy)] + new_params[pos:]
        action = f"insert {component} with strategy {strategy} at position {pos}"

    return new_order, new_params, action

current_order = list(pipeline_order)
current_params = list(best_param)
current_utility = initial_fail_utility

trace_rows = [{
    "iteration": 0,
    "action": "initial failing pipeline",
    "pipeline": show_pipeline_vector(current_order, current_params),
    "utility": current_utility,
    "accepted": True,
    "passes_goal": passes_goal(current_utility, f_goal)
}]

MAX_TUTORIAL_STEPS = 30

for iter_id, intervention in enumerate(fused_ranking[:MAX_TUTORIAL_STEPS], start=1):
    candidate_order, candidate_params, action = apply_single_intervention(
        current_order,
        current_params,
        intervention
    )

    if candidate_order is None:
        trace_rows.append({
            "iteration": iter_id,
            "action": action,
            "pipeline": show_pipeline_vector(current_order, current_params),
            "utility": current_utility,
            "accepted": False,
            "passes_goal": passes_goal(current_utility, f_goal)
        })
        continue

    candidate_utility = fail_executor.current_par_lookup(
        candidate_order,
        candidate_params,
        fixed_data=fixed_failing_data
    )

    improved = better_than(candidate_utility, current_utility)
    if improved:
        current_order = candidate_order
        current_params = candidate_params
        current_utility = candidate_utility

    trace_rows.append({
        "iteration": iter_id,
        "action": action,
        "pipeline": show_pipeline_vector(candidate_order, candidate_params),
        "utility": candidate_utility,
        "accepted": improved,
        "passes_goal": passes_goal(candidate_utility, f_goal)
    })

    if passes_goal(current_utility, f_goal):
        break

df_trace = pd.DataFrame(trace_rows)
display(df_trace)

,iteration,action,pipeline,utility,accepted,passes_goal
0,0,initial failing pipeline,sampling:2 → invalid_value:1 → missing_value:4...,0.134715,True,False
1,1,insert outlier with strategy 2 at position 5,sampling:2 → invalid_value:1 → missing_value:4...,0.110390,True,False
2,2,insert outlier with strategy 5 at position 4,sampling:2 → invalid_value:1 → missing_value:4...,0.113821,False,False
3,3,insert outlier with strategy 4 at position 3,sampling:2 → invalid_value:1 → missing_value:4...,0.113821,False,False
4,4,change missing_value to strategy 2,sampling:2 → invalid_value:1 → missing_value:2...,0.123377,False,False
5,5,change normalization to strategy 4,sampling:2 → invalid_value:1 → missing_value:4...,0.110390,False,False
6,6,change missing_value to strategy 6,sampling:2 → invalid_value:1 → missing_value:6...,0.116883,False,False
7,7,change floating_point to strategy 4,sampling:2 → invalid_value:1 → missing_value:4...,0.110390,False,False
8,8,change missing_value to strategy 8,sampling:2 → invalid_value:1 → missing_value:8...,0.097403,True,True


## Summary of the resolving malfunctioning pipeline

In [39]:
def summarize_pipeline_changes(
    initial_order,
    initial_params,
    final_order,
    final_params,
    ignore_components=("model", "")
):
    changes = []

    initial_pairs = [
        (comp, strat)
        for comp, strat in zip(initial_order, initial_params)
        if comp not in ignore_components and comp is not None
    ]

    final_pairs = [
        (comp, strat)
        for comp, strat in zip(final_order, final_params)
        if comp not in ignore_components and comp is not None
    ]

    initial_map = {}
    for pos, (comp, strat) in enumerate(initial_pairs):
        initial_map[comp] = {
            "position": pos,
            "strategy": strat
        }

    final_map = {}
    for pos, (comp, strat) in enumerate(final_pairs):
        final_map[comp] = {
            "position": pos,
            "strategy": strat
        }

    all_components = sorted(set(initial_map.keys()) | set(final_map.keys()))

    for comp in all_components:
        if comp not in initial_map:
            changes.append({
                "component": comp,
                "change_type": "Inserted",
                "initial_position": None,
                "final_position": final_map[comp]["position"],
                "initial_strategy": None,
                "final_strategy": final_map[comp]["strategy"]
            })

        elif comp not in final_map:
            changes.append({
                "component": comp,
                "change_type": "Deleted",
                "initial_position": initial_map[comp]["position"],
                "final_position": None,
                "initial_strategy": initial_map[comp]["strategy"],
                "final_strategy": None
            })

        else:
            old_pos = initial_map[comp]["position"]
            new_pos = final_map[comp]["position"]
            old_strat = initial_map[comp]["strategy"]
            new_strat = final_map[comp]["strategy"]

            if old_pos != new_pos and old_strat != new_strat:
                change_type = "Moved + Strategy Changed"
            elif old_pos != new_pos:
                change_type = "Moved"
            elif old_strat != new_strat:
                change_type = "Strategy Changed"
            else:
                continue

            changes.append({
                "component": comp,
                "change_type": change_type,
                "initial_position": old_pos,
                "final_position": new_pos,
                "initial_strategy": old_strat,
                "final_strategy": new_strat
            })

    return pd.DataFrame(changes)


changes_df = summarize_pipeline_changes(
    initial_order_,
    initial_params_,
    current_order,
    current_params
)

print("Initial failing utility:", initial_fail_utility)
print("Final utility:", current_utility)
print("Goal:", f_goal)
print("Goal reached?", passes_goal(current_utility, f_goal))
print()

print("Initial pipeline:")
print(show_pipeline_vector(initial_order_, initial_params_))
print()

print("Final pipeline:")
print(show_pipeline_vector(current_order, current_params))
print()

if changes_df.empty:
    print("Changes from initial pipeline: No structural or strategy changes.")
else:
    print("Changes from initial pipeline:")
    display(changes_df)

final_summary = pd.DataFrame([{
    "dataset": dataset_name,
    "metric": metric_type,
    "goal": f_goal,
    "initial_failing_utility":  initial_fail_utility_saved,
    "final_utility": current_utility,
    "goal_reached": passes_goal(current_utility, f_goal),
    "num_evaluated_interventions": len(df_trace) - 1,

}])

display(final_summary)

Initial failing utility: 0.13471502590673579
Final utility: 0.09740259740259738
Goal: 0.1
Goal reached? True

Initial pipeline:
sampling:2 → invalid_value:1 → missing_value:4 → floating_point:1 → normalization:2 → model:1

Final pipeline:
sampling:2 → invalid_value:1 → missing_value:8 → floating_point:1 → normalization:2 → outlier:2 → model:1

Changes from initial pipeline:


,component,change_type,initial_position,final_position,initial_strategy,final_strategy
0,missing_value,Strategy Changed,2.0,2,4.0,8
1,outlier,Inserted,NaN,5,NaN,2


,dataset,metric,goal,initial_failing_utility,final_utility,goal_reached,num_evaluated_interventions
0,hmda,accuracy_score,0.1,0.134715,0.097403,True,8
